# 00 - Phase 0 GATE: checkpoint -> released-score reproduction

**HARD prerequisite.** If the forward pass on LTC's released checkpoint does NOT reproduce
LTC's released softmax scores, **STOP - do not extract embeddings**. This failure is silent
(extraction runs, Phase 1 emits numbers, all meaningless); this check is the only alarm.

Pre-registered criteria: `pcc/reports/phase0_checkpoint_gate.md` (G1 accuracy, G2 NN match,
G3 true-prob curve, G4 label multiset). The check is **permutation-invariant** because LTC's
loaders use `shuffle=True` -> released rows are in an unrecoverable order (see release_audit.md).

## RUN ORDER - read this

**PRE-CHECK A = cells 1-5. Needs NO images.** It downloads only the released scores +
models.zip (small) and answers: does the checkpoint load, does its head dim equal the released
class count, are the released scores internally sane? Run these FIRST and report the output.

**STEP B = cells 6-8. Needs the full image dataset** (Pl@ntNet-300K, ~31 GB, Zenodo record
5645731). Cell 6 detects a missing dataset and SKIPS cleanly; cells 7-8 then stop. That is
expected - it is not an error in the pipeline.


## 1. GPU check


In [ ]:
import subprocess
o = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(o.stdout if o.returncode==0 else 'WARNING: no GPU - forward pass will be slow on CPU.')


## 2. Config - `# === EDIT ME ===`


In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''                       # this repo git URL (or upload manually)
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'

DATASET    = 'plantnet'               # 'plantnet' (start here) | 'inaturalist'
MODEL_TYPE = 'best'                   # best | last-epoch | double-dip
SPLIT      = 'cal'                    # 'cal' or 'test' ONLY. The LTC release has NO 'val'
                                      # split: get_results.py hardcodes split='cal' and the
                                      # README ships only cal + test scores/labels.

# gdown file IDs (from the LTC repo scripts) ------------------------------
GID_MODELS = '1tS-M-4IYyCGMeIxxyrgx2-XCZgdvw18S'   # models.zip (all 6 ResNet-50s)
GID_SCORES = {'plantnet':'1k_PPQV3VJT44hz02CcnbqPstjQo70vGr',
              'inaturalist':'1W8R8Jj2bhS2PbR-3X9vEw-WkanbOk6mq'}

# dataset images: EPHEMERAL /content, NOT Drive -------------------------
# AGENTS.md Sec 3.2: never store images, store embeddings. 31 GB of images on
# Drive would burn quota and be slow to read; /content is fast and disposable.
# If the session dies: re-download (aria2 is fast) and extraction RESUMES from
# the manifest already on Drive.
DATA_ROOT  = '/content/plantnet_300K'      # unzipped Pl@ntNet lives here
ZIP_PATH   = '/content/plantnet_300K.zip'
ZENODO_URL = 'https://zenodo.org/records/5645731/files/plantnet_300K.zip?download=1'
ZENODO_MD5 = 'db27d149f2a6c304b887353c07021687'   # from the Zenodo record API
UNZIP_SPLITS = ('val',)                    # ('val',) for the gate; add 'train' for descriptors
INAT_ANN   = f'{DRIVE_ROOT}/data/inaturalist/val2018.json'   # iNaturalist only

# gate params (defaults match phase0_checkpoint_gate.md - do NOT loosen silently)
SUBSAMPLE      = 3000                 # val images to forward-pass for the gate
NN_SUBSAMPLE   = 1000
TOL_ACC, TOL_NN_LINF, TOL_NN_MEDIAN, TOL_CURVE = 0.002, 1e-4, 1e-5, 1e-3
# G2 (nearest-neighbour row match) needs OUR score columns to share LTC's class
# convention. CONFIRMED for both: LTC's PlantNet subclasses ImageFolder (sorted
# folder names) and iNaturalist uses category_id directly. So G2 is enabled.
CHECK_NN   = True
SEED = 42
# =======================================================================
print('DATASET =', DATASET, '| CHECK_NN(G2) =', CHECK_NN)


## 3. Mount Drive + repo + pinned env + seed + versions


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
REPO_ROOT = os.getcwd()
subprocess.run(['pip','install','-q','gdown','scipy','scikit-learn'], check=False)
os.environ['PYTHONPATH'] = REPO_ROOT + os.pathsep + os.environ.get('PYTHONPATH','')
os.environ['PYTHONUTF8'] = '1'
from pcc.utils.seed import set_seed
from pcc.utils.device import get_device, gpu_name
from pcc.utils.io import environment_stamp
set_seed(SEED); DEVICE = get_device()
print('GPU:', gpu_name()); print('env:', environment_stamp()['packages'])


## 4. Download released checkpoint + scores (idempotent, to Drive)


In [ ]:
import os, glob, subprocess
CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints/ltc_models'
SCORES_DIR = f'{DRIVE_ROOT}/released_scores/{DATASET}'
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(SCORES_DIR, exist_ok=True)

def _has(patt): return len(glob.glob(patt, recursive=True))>0
if not _has(f'{CKPT_DIR}/**/*{DATASET}*model*.pth'):
    subprocess.run(['gdown',GID_MODELS,'-O',f'{CKPT_DIR}/models.zip'], check=True)
    subprocess.run(['unzip','-o',f'{CKPT_DIR}/models.zip','-d',CKPT_DIR], check=True)
if not _has(f'{SCORES_DIR}/**/*{DATASET}*_softmax.npy'):
    subprocess.run(['gdown',GID_SCORES[DATASET],'-O',f'{SCORES_DIR}/{DATASET}.zip'], check=True)
    subprocess.run(['unzip','-o',f'{SCORES_DIR}/{DATASET}.zip','-d',SCORES_DIR], check=True)

CKPT = sorted(glob.glob(f'{CKPT_DIR}/**/{MODEL_TYPE}-{DATASET}-model.pth', recursive=True))
SCF  = sorted(glob.glob(f'{SCORES_DIR}/**/{MODEL_TYPE}-{DATASET}-model_{SPLIT}_softmax.npy', recursive=True))
if not CKPT or not SCF:
    print('--- .pth files found under', CKPT_DIR, '---')
    for f in sorted(glob.glob(f'{CKPT_DIR}/**/*.pth', recursive=True))[:40]: print('   ', f)
    print('--- .npy files found under', SCORES_DIR, '---')
    for f in sorted(glob.glob(f'{SCORES_DIR}/**/*.npy', recursive=True))[:40]: print('   ', f)
    print()
    print('Expected checkpoint pattern:', f'{MODEL_TYPE}-{DATASET}-model.pth')
    print('Expected scores pattern    :', f'{MODEL_TYPE}-{DATASET}-model_{SPLIT}_softmax.npy')
    print('Note: the release provides SPLIT in {cal, test} only - there is no val split.')
assert CKPT, f'checkpoint not found under {CKPT_DIR} (see the listing above)'
assert SCF,  f'released {SPLIT} softmax not found under {SCORES_DIR} (see the listing above)'
CKPT_PATH, SCORES_FOLDER = CKPT[0], os.path.dirname(SCF[0])
print('checkpoint:', CKPT_PATH); print('scores dir:', SCORES_FOLDER)


## 5. PRE-CHECK A - zero images (run first, report back)
Fast alarms needing no dataset download: checkpoint loads, head dim == released #classes,
released scores internally sane (their own argmax accuracy high).


In [ ]:
import numpy as np
from pcc.data.ltc_datasets import load_released_scores, NUM_CLASSES
from pcc.extract.backbones import load_ltc_resnet50
from pcc.eval.score_repro import top1_accuracy, sha256_file

rel_softmax, rel_labels = load_released_scores(SCORES_FOLDER, DATASET, SPLIT, MODEL_TYPE)
print(f'using released split = {SPLIT!r} (release has cal + test only)')
C_rel = rel_softmax.shape[1]; acc_rel = top1_accuracy(rel_softmax, rel_labels)
print(f'released {SPLIT}: N={len(rel_labels)} C={C_rel} self-accuracy={acc_rel:.4f} '
      f'labels[min,max]=[{rel_labels.min()},{rel_labels.max()}]')
assert C_rel == NUM_CLASSES[DATASET], f'released C {C_rel} != expected {NUM_CLASSES[DATASET]}'

model = load_ltc_resnet50(CKPT_PATH, NUM_CLASSES[DATASET], DEVICE)
head = model.fc.out_features
print('checkpoint fc out_features =', head)
assert head == C_rel, f'HEAD DIM {head} != released C {C_rel} -> wrong checkpoint/arch (STOP)'
print('PRE-CHECK A OK: head matches released classes; released scores sane.')
print('ckpt sha256:', sha256_file(CKPT_PATH)[:16], '...')


## 5b. PRE-CHECK A2 - per-class calibration counts (still ZERO images)

The released labels alone answer two questions that gate Phase 1, at no cost:

- **Which alpha are computable per class?** A classwise conformal quantile is finite only if
  `ceil((n+1)(1-alpha)) <= n`, i.e. `n >= ceil(1/alpha) - 1`: **n>=9 for alpha=0.1, n>=19 for
  0.05, n>=99 for 0.01**. Classes below that get an infinite threshold (the whole label space)
  and fall back per reports/fallback_policy.md.
- **What does matched-n cost?** Amendment 2 needs a common `n_cal` per class; every class with
  fewer samples is DROPPED, and dropping tail classes is itself prevalence-linked selection.
  This table is the input to that decision.


In [ ]:
import numpy as np
counts = np.bincount(rel_labels, minlength=C_rel)
print(f'classes: {C_rel} | with >=1 cal sample: {(counts>0).sum()} | '
      f'EMPTY (delta_y undefined): {(counts==0).sum()}')
qs = [0, 1, 5, 10, 25, 50, 75, 90, 100]
print('per-class count percentiles:',
      {f'p{q}': int(np.percentile(counts, q)) for q in qs})
print()
print('multi-alpha feasibility (Sec 8.8) - classwise quantile finite only if n >= ceil(1/a)-1:')
for a in (0.01, 0.05, 0.1):
    need = int(np.ceil(1/a)) - 1
    ok = int((counts >= need).sum())
    print(f'  alpha={a:<5} needs n>={need:<3} -> {ok:5d}/{C_rel} classes ({100*ok/C_rel:5.1f}%)')
print()
print('matched-n cost (Amendment 2) - classes retained at each n_cal:')
for nc in (5, 10, 20, 25, 50, 100, 200):
    keep = int((counts >= nc).sum())
    frac_data = float(counts[counts>=nc].clip(max=nc).sum()) / max(counts.sum(), 1)
    print(f'  n_cal={nc:<4} -> {keep:5d}/{C_rel} classes kept ({100*keep/C_rel:5.1f}%), '
          f'{100*frac_data:5.1f}% of cal samples used')
print()
print('head/tail split (prevalence quartiles of the retained classes):')
nz = counts[counts > 0]
print(f'  tail quartile (<=p25): n<={int(np.percentile(nz,25))} | '
      f'head quartile (>=p75): n>={int(np.percentile(nz,75))}')


### === PRE-CHECK A ENDS ===
For the **pre-check-only** run (both datasets): stop here and report the printed output
(released N/C/self-accuracy, head_out_features, sha256) before running Step B. Step B
downloads the image dataset; do not start it until the gate target is chosen.


## 5c. Download the image dataset with aria2c (only needed for STEP B)

One file: `plantnet_300K.zip`, **31.67 GB**, MD5 `db27d149f2a6c304b887353c07021687`.

- **aria2c with 16 connections** is far faster than wget/curl on Zenodo, and resumes
  (`-c`) if the transfer drops.
- **The MD5 is verified by aria2c during the download.** A silently truncated 32 GB file
  would corrupt every downstream number with nothing to reveal it, so this is not optional.
- **Selective unzip.** `UNZIP_SPLITS=('val',)` extracts only `images/val/*` (~3 GB of the
  306k images) which is all the checkpoint gate needs. Add `'train'` later for descriptors.

Everything here writes to **/content (ephemeral)**. Only embeddings go to Drive.


In [ ]:
import os, subprocess, shutil

free_gb = shutil.disk_usage('/content').free / 1e9
print(f'/content free space: {free_gb:.1f} GB')
need = 33 + (33 if 'train' in UNZIP_SPLITS else 4)
if free_gb < need:
    print(f'WARNING: about {need} GB needed (zip + unzipped splits). Consider')
    print('  keep UNZIP_SPLITS limited to val, and delete the zip after unzipping.')

subprocess.run(['apt-get', '-qq', 'install', '-y', 'aria2'], check=False)

if not os.path.exists(ZIP_PATH):
    cmd = ['aria2c', '-x16', '-s16', '-k1M', '-c',
           '--console-log-level=warn', '--summary-interval=30',
           f'--checksum=md5={ZENODO_MD5}',
           '-d', os.path.dirname(ZIP_PATH), '-o', os.path.basename(ZIP_PATH),
           ZENODO_URL]
    print('running:', ' '.join(cmd[:6]), '...')
    r = subprocess.run(cmd)
    if r.returncode != 0:
        raise SystemExit(f'aria2c failed (exit {r.returncode}). Re-run this cell - it resumes.')
else:
    print('zip already present:', ZIP_PATH,
          f'({os.path.getsize(ZIP_PATH)/1e9:.1f} GB)')

# Selective unzip. -n = never overwrite, so re-running is cheap and idempotent.
for sp in UNZIP_SPLITS:
    target = f'{DATA_ROOT}/images/{sp}'
    if os.path.isdir(target) and any(os.scandir(target)):
        print(f'already unzipped: {target}'); continue
    print(f'unzipping images/{sp} ...')
    subprocess.run(['unzip', '-n', '-q', ZIP_PATH,
                    f'plantnet_300K/images/{sp}/*', '-d', '/content'], check=True)

for sp in UNZIP_SPLITS:
    d = f'{DATA_ROOT}/images/{sp}'
    n_cls = len([e for e in os.scandir(d) if e.is_dir()]) if os.path.isdir(d) else 0
    print(f'  images/{sp}: {n_cls} class folders')
print()
print('Free space now: %.1f GB. Deleting the zip frees ~32 GB once unzipping is done.' %
      (shutil.disk_usage('/content').free / 1e9))
print('  os.remove(ZIP_PATH)   # run this manually when you are sure')


## 6. Build the dataset for the released split (STEP B - NEEDS THE IMAGES)

**Do not run this until the dataset is actually downloaded.** Creating an empty folder to
get past a path error does not help - ImageFolder will just fail on 'no class folder'.

Two things that are easy to get wrong, both handled by `pcc.data.ltc_datasets`:

1. **Score-split names are not directory names.** `PlantNet.split_folder` is
   `os.path.join(root, split)` with split in {train, val, test} - there is **no `cal`
   directory**. The released `cal` AND `val` arrays both come from the **val** directory.
2. **`cal` is a reproducible 70% subset of val.** LTC does
   `np.random.seed(0); shuffle(indices)`, taking the first 30% as proper-val and the rest as
   cal. Membership is therefore exactly reconstructable, which is what makes N and the label
   multiset (G4) a real check. Row ORDER is still lost (loaders use shuffle=True), so the gate
   stays permutation-invariant.

`PlantNet` subclasses `ImageFolder`, so the class-index convention IS ImageFolder's sorted
folder names - which is why G2 is now enabled for Pl@ntNet (it used to be skipped).


In [ ]:
import os, numpy as np
from torch.utils.data import DataLoader, Subset
from pcc.data.ltc_datasets import (test_transform, plantnet_scored_subset,
                                   INaturalist2018Val, NUM_CLASSES)

# NOTE: this cell never raises. It sets STEP_B_READY; cells 7-8 skip cleanly if False.
# (Raising SystemExit from inside an `except` block trips an IPython traceback bug and
#  buries the actual message in noise.)
STEP_B_READY = False
ds = None
tfm = test_transform()

if DATASET == 'plantnet':
    try:
        ds, ds_labels, class_to_idx = plantnet_scored_subset(DATA_ROOT, SPLIT, transform=tfm)
        print('class_to_idx sample:', dict(list(class_to_idx.items())[:5]))
        print(f'classes found: {len(class_to_idx)} (expect {NUM_CLASSES[DATASET]})')
        STEP_B_READY = True
    except FileNotFoundError as e:
        reason = str(e)
else:
    if os.path.exists(INAT_ANN):
        ds = INaturalist2018Val(f'{DATA_ROOT}/', INAT_ANN, transform=tfm)
        STEP_B_READY = True
    else:
        reason = f'missing {INAT_ANN} (see release_audit.md iNaturalist logistics)'

if not STEP_B_READY:
    print('=' * 74)
    print('STEP B SKIPPED - the image dataset is not present. This is expected if you are')
    print('only running PRE-CHECK A (cells 1-5), which needs NO images.')
    print('=' * 74)
    print(reason)
    print()
    print('Expected layout (Pl@ntNet-300K, Zenodo record 5645731):')
    print(f'  {DATA_ROOT}/images/train/<class>/*.jpg')
    print(f'  {DATA_ROOT}/images/val/<class>/*.jpg   <- cal AND val scores come from here')
    print(f'  {DATA_ROOT}/images/test/<class>/*.jpg')
    print()
    print('An EMPTY folder does not help - ImageFolder needs populated class sub-folders.')
else:
    print(f'released {SPLIT} rows = {len(rel_labels)} | reconstructed subset = {len(ds)}')
    if len(ds) != len(rel_labels):
        print('WARNING: size mismatch -> G4 will fail. Check DATA_ROOT and the split mapping.')
    rng = np.random.default_rng(SEED)
    sub = rng.choice(len(ds), min(SUBSAMPLE, len(ds)), replace=False)
    loader = DataLoader(Subset(ds, sub.tolist()), batch_size=64, shuffle=False, num_workers=2)
    print(f'{DATASET} {SPLIT}: pool={len(ds)} forward-pass subsample={len(sub)}')


## 7. Forward pass -> our scores (float64 logits -> scipy softmax, exactly like LTC)


In [ ]:
from pcc.extract.backbones import forward_logits_and_embeddings
from pcc.eval.score_repro import top1_accuracy
if not STEP_B_READY:
    print('skipped: Step B not ready (see cell 6). Nothing to forward-pass.')
    mine_softmax = mine_labels = None
else:
    mine_softmax, mine_labels, _ = forward_logits_and_embeddings(
        model, loader, DEVICE, capture_embeddings=False)
    print('our subsample accuracy:', round(top1_accuracy(mine_softmax, mine_labels),4),
          '| released self-accuracy:', round(acc_rel,4))


## 8. EVALUATE GATE + STOP on FAIL + write report/marker


In [ ]:
import time, json, os
from pcc.eval.score_repro import evaluate_gate, sha256_file
from pcc.utils.io import write_report

if not STEP_B_READY:
    print('skipped: Step B not ready (see cell 6). No gate verdict computed.')
else:
    res = evaluate_gate(mine_softmax, mine_labels, rel_softmax, rel_labels,
                        nn_subsample=NN_SUBSAMPLE, seed=SEED, tol_acc=TOL_ACC,
                        tol_nn_linf=TOL_NN_LINF, tol_nn_median=TOL_NN_MEDIAN,
                        tol_curve=TOL_CURVE, check_nn=CHECK_NN)
    print(json.dumps(res, indent=2))
    
    checksums = {'checkpoint': sha256_file(CKPT_PATH)}
    report = write_report('pcc/reports', f'00_verify_checkpoint_{DATASET}_{SPLIT}',
        hypothesis='LTC released resnet50 reproduces released softmax on '+DATASET+' '+SPLIT,
        pass_criteria='G1 |acc|<=0.002; G2 NN>=99%@1e-4 (iNat only); G3 curve<=1e-3; G4 label multiset',
        config=dict(dataset=DATASET, split=SPLIT, model_type=MODEL_TYPE, subsample=int(len(sub)),
                    check_nn=CHECK_NN, tolerances=dict(acc=TOL_ACC, nn_linf=TOL_NN_LINF,
                    nn_median=TOL_NN_MEDIAN, curve=TOL_CURVE)),
        seed=SEED, results={**res, 'checksums':checksums, 'released_self_accuracy':acc_rel},
        conclusion=res['verdict'], started_at=time.time())
    print('report:', report)
    
    GATE_MARKER = f'{DRIVE_ROOT}/gates/GATE_PASSED_{DATASET}_{SPLIT}.json'
    os.makedirs(os.path.dirname(GATE_MARKER), exist_ok=True)
    if res['verdict'] == 'PASS':
        with open(GATE_MARKER,'w') as f:
            json.dump({'dataset':DATASET,'split':SPLIT,'checksums':checksums,'results':res}, f, indent=2)
        print('GATE PASSED - marker written. Extraction (01) may proceed.')
    else:
        if os.path.exists(GATE_MARKER): os.remove(GATE_MARKER)
        raise SystemExit('GATE FAILED - STOP. Do NOT extract. Report results above. '
                         'Likely cause: transform/normalize/checkpoint/class-index convention.')
